# Day 96 — Image Classifier

This notebook covers Multi-Class Image Classification Pipeline, `nn.CrossEntropyLoss()` & `torch.long` Target Conventions, Structured Synthetic Image Dataset (`SyntheticPatternImageDataset`), End-to-End `SimpleCNN` Training & Evaluation (`train_epoch`, `eval_epoch`), and Logit Calibration vs. Accuracy D

## Validation goals
- Implement `SyntheticPatternImageDataset` generating `3x32x32` `torch.float32` image tensors with four distinct spatial patterns (horizontal band, vertical band, central block, diagonal stripe) and `torch.int64` (`torch.long`) class indices
- Build sample-weighted `train_epoch` and `eval_epoch` routines computing multi-class cross-entropy loss (`nn.CrossEntropyLoss()`) and top-1 classification accuracy (`logits.argmax(dim=1)`)
- Train `SimpleCNN(in_channels=3, num_classes=4, img_size=32, channel_dims=(16, 32, 64), hidden_fc=64, dropout=0.1)` across 12 epochs using `Adam(lr=3e-3, weight_decay=1e-4)
- Analyze why `nn.CrossEntropyLoss()` requires `(N, num_classes)` raw logits and `torch.long` targets, how `logits.argmax(dim=1)` maps scores to class indices, and why validation loss can rise while top-1 accuracy stays stable

### 1. Classification Loss and Accuracy Mechanics

Unlike regression (MSELoss predicting continuous values), multi-class image classification predicts discrete class indices (0 to num_classes - 1):
- Loss Function (`nn.CrossEntropyLoss()`): Expects logits of shape (N, num_classes) (float32) and integer targets of shape `(N,)` `(torch.long / int64)`. It applies `log-softmax` and negative log-likelihood internally in a numerically stable way—so keep your model's final layer linear without Softmax.
- Predicted Class `(preds.argmax(dim=1))`: Picks the index with the highest logit per sample.
- Accuracy: Fraction of samples where `pred_class == target_class`.

### 2. Building a Self-Contained Image Dataset & Classifier Pipeline

When torchvision datasets (such as `CIFAR-10 or FashionMNIST`) are available locally or via download, you can plug them directly into `DataLoaders`. To ensure this script runs reliably offline or in constrained environments without external downloads, the module below includes a deterministic structured synthetic image dataset (distinct geometric/intensity patterns per class) with a clean hook to swap in `torchvision` datasets when available.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from day_95_cnn_architecture import SimpleCNN


def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class SyntheticPatternImageDataset(Dataset):
    """Generates reproducible 3x32x32 image tensors with class-specific visual structure."""

    def __init__(
        self,
        num_samples: int = 600,
        num_classes: int = 4,
        img_size: int = 32,
        seed: int = 42,
    ):
        rng = np.random.default_rng(seed)
        images = np.zeros((num_samples, 3, img_size, img_size), dtype=np.float32)
        labels = np.zeros(num_samples, dtype=np.int64)

        for i in range(num_samples):
            cls = int(i % num_classes)
            labels[i] = cls

            # Base mild background noise
            img = rng.normal(0.0, 0.15, size=(3, img_size, img_size)).astype(np.float32)

            if cls == 0:
                # Bright horizontal band across middle rows
                img[0, 12:20, :] += 1.4
            elif cls == 1:
                # Bright vertical band across middle columns
                img[1, :, 12:20] += 1.4
            elif cls == 2:
                # Central square block
                img[2, 10:22, 10:22] += 1.5
            else:
                # Diagonal stripe pattern
                for idx in range(img_size):
                    img[0, idx, idx] += 1.4
                    img[1, idx, (idx + 1) % img_size] += 1.0

            images[i] = img

        self.X = torch.from_numpy(images)
        self.y = torch.from_numpy(labels)

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    model.train()
    loss_sum = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_size = batch_x.shape[0]

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            pred_classes = logits.argmax(dim=1)
            correct += int((pred_classes == batch_y).sum().item())
            loss_sum += loss.item() * batch_size
            total += batch_size

    return loss_sum / total, correct / total


def eval_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_size = batch_x.shape[0]

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            pred_classes = logits.argmax(dim=1)
            correct += int((pred_classes == batch_y).sum().item())
            loss_sum += loss.item() * batch_size
            total += batch_size

    return loss_sum / total, correct / total


if __name__ == "__main__":
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    num_classes = 4
    img_size = 32

    train_ds = SyntheticPatternImageDataset(num_samples=600, num_classes=num_classes, img_size=img_size, seed=42)
    val_ds = SyntheticPatternImageDataset(num_samples=200, num_classes=num_classes, img_size=img_size, seed=99)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)

    model = SimpleCNN(
        in_channels=3,
        num_classes=num_classes,
        img_size=img_size,
        channel_dims=(16, 32, 64),
        hidden_fc=64,
        dropout=0.1,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)

    epochs = 12
    print("--- Day 96: CNN Image Classifier ---")
    print(f"Device: {device} | Classes: {num_classes} | Train: {len(train_ds)} | Val: {len(val_ds)}\n")

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = eval_epoch(model, val_loader, criterion, device)

        if epoch == 1 or epoch % 3 == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:2d}/{epochs:2d} | "
                f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc * 100:6.2f}% | "
                f"Val Loss: {va_loss:.4f} | Val Acc: {va_acc * 100:6.2f}%"
            )

### 3. Practical Checks for Image Classifiers

- Target Dtype (`torch.long`): `nn.CrossEntropyLoss()` raises a RuntimeError if batch_y is float32 instead of integer int64 (`torch.long`).
- Loss vs. Accuracy: Loss measures confidence and margin across all classes (via `log-softmax`), while accuracy measures top-1 hard decisions. Validation loss can occasionally rise slightly even when accuracy holds steady if predictions grow overconfident on wrong guesses.
- Class Balance: Always check class counts in train and validation splits—class imbalance can inflate overall accuracy while hiding poor recall on minority classes.

### Guided Question:

Why does `nn.CrossEntropyLoss()` expect raw logits of shape (`N, num_classes`) and integer class indices of dtype torch.long (`int64`)?

`nn.CrossEntropyLoss()` expects logits of shape (`N, num_classes`) and integer class indices of dtype torch.long (`int64`) because of how PyTorch structures batched classification math, memory indexing, and numerical stability.

**1. Why Logits Have Shape (N, num_classes)**

- Batch dimension (N): N is the number of samples in the mini-batch. Each row i (from 0 to N - 1) holds the score vector for one sample across all C = num_classes categories.
- Class dimension (num_classes): Softmax normalizes across classes for each sample independently, so dimension 1 must contain all candidate class scores for a single item.
- Fused loss evaluation: nn.CrossEntropyLoss() computes LogSoftmax along the class axis and then selects the log-probability of each sample's target class. (For higher-dimensional inputs like spatial segmentation maps (N, C, H, W), PyTorch treats dimension 1 as the class axis and flattens or reduces over the remaining dimensions.)

### 2. Why Targets Are Integer Indices (torch.long / int64)

- Direct tensor indexing (gather / NLLLoss): In NLLLoss, the loss for sample i with true class y_i is simply -log_softmax(z_i)[y_i]. An integer index directly selects the target entry along the class dimension rather than multiplying a full one-hot vector across all C classes.
- Lower memory and bandwidth: A 1D index tensor of shape (N,) uses far less memory than a dense (N, num_classes) one-hot tensor, especially when num_classes is large (such as tens of thousands of tokens in language modeling).
- Built-in label smoothing and ignore_index: Integer targets map cleanly to sentinel values like ignore_index=-100 (commonly used for padded sequence positions) and support label_smoothing without requiring pre-smoothed target distributions.

**Why torch.long (int64) Specifically**

PyTorch indexing convention: Tensor indexing operations and `C++/CUDA` index kernels in PyTorch standardize on signed 64-bit integers (`int64 / torch.long`) for indices and shapes to avoid overflow and type mismatches across backends.

Preventing accidental float targets: Passing floating-point tensors (float32 or float64) as class indices is disallowed because floats are continuous values rather than discrete array indices, whereas class probabilities (soft targets) are passed as a floating-point tensor of shape (N, num_classes) when using soft labels.

### Guided Question:

Given a logits tensor of shape (32, 10) from a 10-class image classifier, how does `logits.argmax(dim=1)` produce class predictions, and what is the resulting shape?

### Answer:

`logits.argmax(dim=1)` scans across the 10 class scores for each of the 32 samples independently and returns the column index of the highest logit in each row, producing a tensor of shape (32,) with dtype torch.int64 (`torch.long`).

**1. How dim=1 Reduction Works**

- Input shape (32, 10): Dimension 0 (32) indexes each sample in the batch; dimension 1 (10) indexes the 10 class logits for a single sample.
- Reduction along dim=1: For each row i (from 0 to 31), PyTorch compares the 10 values logits[i, 0], logits[i, 1], ..., logits[i, 9] and records the index c where logits[i, c] reaches its maximum.
- Resulting shape (32,): Because dimension 1 is reduced to a single index per row (with keepdim=False by default), the output is a 1D tensor of length 32, where entry i is the predicted class index (0 through 9) for sample i.
- Tie-breaking rule: If two or more classes share the exact maximum logit in a row, argmax returns the first (lowest) index that achieves that maximum.

**2. Why argmax on Raw Logits Matches argmax on Softmax**
Softmax is a strictly monotonically increasing transformation along the class dimension for any given sample: if logit $a > \text{logit } b$, then $\text{softmax}(z)_a > \text{softmax}(z)_b$. Computing argmax(dim=1) directly on raw logits gives the exact same predicted class indices as running torch.softmax(logits, dim=1).argmax(dim=1), while skipping unnecessary exponentiation and normalization work.

### Guided Question:

How can validation accuracy remain high or stable while validation loss starts ticking upward?

### Answer:

Validation accuracy can stay high or flat while validation loss climbs because accuracy is a threshold metric (only caring which class has the highest score), whereas cross-entropy loss penalizes confidence and probability calibration across all classes. A model can get the right answer more often—or just as often—while becoming overconfident on its mistakes or less confident on its correct predictions.

**1. Overconfidence on Wrong Predictions**

Cross-entropy loss scales with $-\log p_y$, where $p_y$ is the predicted probability for the true class $y$.
- If a sample is misclassified with probability $0.60$ on a wrong class and $0.40$ on the true class, loss is $-\log(0.40) \approx 0.92$.
- As training pushes weights to larger magnitudes, that same wrong prediction might shift to $0.95$ on the wrong class and $0.05$ on the true class.
- Accuracy does not change (it was already wrong), but loss jumps from $\approx 0.92$ to $-\log(0.05) \approx 3.00$. Just a few high-confidence errors can lift average batch loss noticeably.

**2. Eroding Margins on Correct Predictions**
Accuracy only checks whether $\arg\max_c z_c = y$; it ignores how far ahead the winning logit $z_y$ is over runner-up logits.
Suppose a model correctly predicts class $y$ with logits $[4.0, 0.1, -1.0]$ early in training, and later those logits shift to $[1.6, 1.3, 0.5]$.The top class is still $y$, so accuracy stays unchanged, but the Softmax probability for class $y$ drops because probability mass spreads toward the runner-up class—raising cross-entropy loss.

**3. Early Overfitting on Hard or Ambiguous Examples**

Many validation examples may already be easy and cleanly separated, keeping accuracy steady.

Meanwhile, a subset of noisy, borderline, or visually ambiguous validation samples begins to be memorized or over-sharpened by the network, causing sharp spikes in loss on those specific items without flipping enough overall predictions to drop accuracy.

**4. Logit Magnitude Growth (Weight Norm / Temperature Effect)**
Without strong weight decay or normalization constraints, raw logits often grow larger in magnitude over epochs even when decision boundaries barely move.

- Larger logits make Softmax distributions sharper ("more extreme"), amplifying loss on any imperfect or mislabeled validation example while leaving argmax decisions largely intac

**Practical Takeaway**

When validation accuracy holds steady or rises while validation loss creeps up, check whether the model is losing calibration rather than failing catastrophically on class boundaries. Techniques like weight decay, dropout, label smoothing, early stopping on validation loss, or temperature scaling (post-hoc calibration) help keep probabilities well-behaved.